In [ ]:
import pandas as pd
from pathlib import Path

INPUT = Path("eeg_cleaned.csv")
OUTPUT = Path("EEG_Limpio_definitibo.csv")

REST = ["Rest_C", "Rest_O"]
TEMP_REMOVE = ["USER_10_CB2", "USER_39_CB2"]
FIX_DUP_USERS = ["USER_13_CB", "USER_08_CB", "USER_09_CB2", "USER_38_CB2"]


def normalize_avatar_order(df):
    """
    Simplifica los nombres de avatar.

    Neutral se trata por orden dentro de cada usuario:
    - primera conversación neutral  -> Neutral1
    - segunda conversación neutral -> Neutral2

    """
    df = df.copy()

    # Convertir cualquier variante neutral a una etiqueta temporal común.
    neutral_mask = df["avatar"].astype(str).str.startswith("Neutral")
    df.loc[neutral_mask, "avatar"] = "Neutral"

    # Simplificar emociones no neutrales si vinieran con nombres largos.
    emotion_map = {
        "Relax": "Relax",
        "Angry": "Angry",
        "Sad": "Sad",
        "Happy": "Happy",
    }
    for emotion in emotion_map:
        mask = df["avatar"].astype(str).str.startswith(emotion)
        df.loc[mask, "avatar"] = emotion_map[emotion]

    # Asignar Neutral1 / Neutral2 según el orden de aparición dentro de cada usuario.
    neutral_mask = df["avatar"].eq("Neutral")
    neutral_order = df.loc[neutral_mask].groupby("user").cumcount() + 1
    df.loc[neutral_mask, "avatar"] = "Neutral" + neutral_order.astype(str)

    return df


def expected_avatar_counts():
    return pd.Series({
        "Neutral1": 1,
        "Neutral2": 1,
        "Relax": 1,
        "Angry": 1,
        "Sad": 1,
        "Happy": 1,
    })


orig = pd.read_csv(INPUT)

#    Se eliminan Rest_C y Rest_O 
cb2 = orig[orig["user"].isin(TEMP_REMOVE) & ~orig["avatar"].isin(REST)].copy()
cb2 = normalize_avatar_order(cb2)


df = orig[~orig["user"].isin(TEMP_REMOVE) & ~orig["avatar"].isin(REST)].copy()
df = normalize_avatar_order(df)


for u in FIX_DUP_USERS:
    m = (df["user"] == u) & (~df["avatar"].isin(["Neutral1", "Neutral2"]))
    sub = df.loc[m]
    occ = sub.groupby("avatar").cumcount() + 1
    total = sub.groupby("avatar")["avatar"].transform("size")
    keep = (total.eq(1) & occ.eq(1)) | (total.gt(1) & occ.eq(2))
    df = df.drop(sub.index[~keep])


# Este usuario originalmente tenia 5 conversaciones, se decide imoputar copiando su otra conversacion neutral
existing_neutrals = set(df.loc[df["user"] == "USER_10_CB", "avatar"])
missing_neutrals = [n for n in ["Neutral1", "Neutral2"] if n not in existing_neutrals]

if missing_neutrals:
    row = df[
        (df["user"] == "USER_10_CB") &
        (df["avatar"].isin(["Neutral1", "Neutral2"]))
    ].iloc[0].copy()
    row["avatar"] = missing_neutrals[0]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

# Este usuario tambien solo 5 conversacciones, se imputa la conversacion de Sad con la mediana de los sujetos PHQ=2 en Sad.
sad_phq2 = df[(df["phq"] == 2) & (df["avatar"] == "Sad")]
row = df[df["user"] == "USER_50_CB"].iloc[0].copy()
row["avatar"] = "Sad"

for col in df.select_dtypes(include="number").columns:
    if col not in ["phq", "label"]:
        row[col] = sad_phq2[col].median()

row["phq"] = df.loc[df["user"] == "USER_50_CB", "phq"].iloc[0]
row["label"] = df.loc[df["user"] == "USER_50_CB", "label"].iloc[0]
df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


#Este usuario tenia 2 coversaciones etiquetadas como Sad y le falta una conversacion como Angry,
#probablemente por un error de etiquetado, se realiza un PCA de todas las conversaciones de SAD,
# Se proyectan en 2D (Archivo de Dragonet) y se ve que la segundo SAD difiere claramente del resto por lo que se decide cambiar su etiqueta a Angry.
idx = df[(df["user"] == "USER_27_CB") & (df["avatar"] == "Sad")].index
if len(idx) >= 2:
    df.loc[idx[1], "avatar"] = "Angry"


df = pd.concat([df, cb2], ignore_index=True)


df.to_csv(OUTPUT, index=False)

print(f"Guardado: {OUTPUT}")
print(f"Filas: {len(df)} | Columnas: {df.shape[1]} | Usuarios: {df['user'].nunique()}")

print("\nUsuarios con nº de conversaciones != 6:")
conv_por_user = df.groupby("user").size()
print(conv_por_user.loc[lambda s: s.ne(6)].to_string())

print("\nUsuarios con combinación de avatares distinta a la esperada, considerando solo usuarios con 6 filas:")
expected = expected_avatar_counts()
bad = []
for u, g in df.groupby("user"):
    if len(g) == 6:
        counts = g["avatar"].value_counts().reindex(expected.index, fill_value=0)
        if not counts.equals(expected):
            bad.append(u)

print(bad if bad else "Ninguno")

print("\nAvatares finales:")
print(df["avatar"].value_counts().to_string())


Guardado: EEG_Limpio_definitibo.csv
Filas: 558 | Columnas: 31 | Usuarios: 94

Usuarios con nº de conversaciones != 6:
user
USER_10_CB2    3
USER_39_CB2    3

Usuarios con combinación de avatares distinta a la esperada, considerando solo usuarios con 6 filas:
Ninguno

Avatares finales:
avatar
Neutral1    94
Sad         94
Happy       94
Neutral2    92
Relax       92
Angry       92
